In [1]:
import os
import pandas as pd
from pathlib import Path
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (NTXpred)

This notebook curates the **NTXpred** source dataset. The raw file is provided in a FASTA-like format with non-standard header/line structure, so it is first repaired into a valid FASTA representation and then parsed into a standardized table. All sequences are treated as **neurotoxic positives**, duplicate consistency checks are applied, and the curated dataset and metadata are exported for downstream analysis.

- **Toxic effect / endpoint:** neurotoxic
- **Source:** NTXpred
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Parses a non-standard FASTA file** (`NTXpred.fa`) using a custom loader (`read_fasta_ntxpred`) that:
  - reconstructs headers that may be split across multiple lines,
  - keeps only valid amino-acid sequence lines,
  - outputs a clean table with `id`, `description`, and `sequence`.
- **Assigns labels**:
  - sets `label = 1` for all entries (positive-only neurotoxic set).
- **Keeps a standardized schema**:
  - `sequence`
  - `label`
- **Checks duplicated sequences**:
  - identical sequences are collapsed,
  - any unexpected inconsistencies are reported as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated outputs**:
  - `processed_neurotoxic_dataset.csv`,
  - `metadata.json`.

In [2]:
name_source = "NTXpred"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df = (
    read_fasta_ntxpred(f"{PATH_INPUT}/{name_source}/NTXpred.fa")
    .assign(label=1)
    [["sequence", "label"]]
)
df.shape

(932, 2)

- Checking duplicates

In [4]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [5]:
df_full.shape

(916, 2)

In [6]:
df_errors.shape

(0, 1)

- Working with metada

In [7]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [8]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2007,
 'last update date': datetime.datetime(2007, 11, 1, 0, 0),
 'download date': Timestamp('2024-07-01 00:00:00'),
 'file format': 'fasta',
 'peptide property': 'neurotoxic, toxic',
 'dataset information': 'Positive',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'No information',
 'repository or server': 'www.imtech.res.in/raghava/ntxpred/',
 'publication': 'https://journals.sagepub.com/doi/abs/10.3233/ISI-2007-00295',
 'number_of_raw_sequences': 932,
 'number_of_sequences_retained': 916,
 'number_of_positive_sequences': 916,
 'number_of_negative_sequences': 0,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [9]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [10]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_neurotoxic_dataset.csv", index=False)